In [18]:
import pandas as pd
seq_regions = ['SEQ_H1', 'SEQ_H2', 'SEQ_L1', 'SEQ_L2', 'SEQ_L3']
cf_regions = ['CF_H1', 'CF_H2', 'CF_L1', 'CF_L2', 'CF_L3']
df = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset=seq_regions)
    .dropna(subset=cf_regions)
    .drop_duplicates(subset=seq_regions) 
)
antigen_counts = df["antigen_name"].value_counts() # Tabelle aus antigen_names und ihren Häufigkeiten in der Spalte antigen_name
df = df[df["antigen_name"].isin(antigen_counts[antigen_counts >= 5].index)] # Behält nur Zeilen, deren antigen_name mindestens 5-mal vorkommt

In [19]:
#da in mmseq2 tabelle viele missing values waren (nicht anwendbar auf länge 5) deswegen hier nochmal gefiltert
#aussagekraft ????

import pandas as pd
from sklearn.metrics import v_measure_score

# Liste der CDR-Regionen
cdrs = ["H1", "H2", "L1", "L2", "L3"]

# Originale und vorhergesagte Clusterlabels einlesen
df_true = df
df_pred = pd.read_csv("data/MMseqs2/MMseqs2_summary_cluster.tsv", sep="\t")

for cdr in cdrs:
    col = f"CF_{cdr}"

    true_labels = df_true[col]
    pred_labels = df_pred[col]

    # Nur gültige (nicht-NaN) Einträge behalten                    if CDR sequence Länge 5 war--->kein clustering-->missing values--> vmeasure funktioniert nicht
    mask = true_labels.notna() & pred_labels.notna()
    true_filtered = true_labels[mask]
    pred_filtered = pred_labels[mask]

    if len(true_filtered) == 0:
        print(f"⚠️  Keine gemeinsamen Werte für {cdr}, übersprungen.")
        continue

    # V-Measure berechnen
    v_score = v_measure_score(true_filtered, pred_filtered)

    print(f"CDR {cdr}: V-Measure = {v_score:.3f} (n = {len(true_filtered)})")


CDR H1: V-Measure = 0.300 (n = 128)
CDR H2: V-Measure = 0.417 (n = 94)
CDR L1: V-Measure = 0.386 (n = 128)
CDR L2: V-Measure = 0.000 (n = 128)
CDR L3: V-Measure = 0.417 (n = 126)


In [21]:
import pandas as pd
from sklearn.metrics import v_measure_score

# Liste der CDR-Regionen
cdrs = ["H1", "H2", "L1", "L2", "L3"]

# DataFrames vorbereiten: 
# df_true enthält die Spalte "pdb", df_pred die Spalte "pdb" bzw. "PDB_ID"
# Falls df_pred in Deiner Realität "PDB_ID" heißt, ersetze hier "pdb" durch "PDB_ID" im p-Subset.
for cdr in cdrs:
    col = f"CF_{cdr}"
    
    # Spaltenauswahl und Umbenennung
    t = df_true[["pdb", col]].rename(columns={col: f"{col}_true"})
    p = df_pred[["pdb", col]].rename(columns={col: f"{col}_pred"})
    
    # Merge auf pdb
    merged = t.merge(p, on="pdb", how="inner")
    
    # Nur Zeilen, in denen sowohl true als auch pred nicht NaN sind
    mask = merged[f"{col}_true"].notna() & merged[f"{col}_pred"].notna()
    n    = mask.sum()
    
    if n == 0:
        print(f"⚠️  Keine gemeinsamen Werte für {cdr}, übersprungen.")
        continue
    
    # Filtere und berechne V-Measure
    true_vals = merged.loc[mask, f"{col}_true"]
    pred_vals = merged.loc[mask, f"{col}_pred"]
    v_score   = v_measure_score(true_vals, pred_vals)
    
    print(f"CDR {cdr}: V-Measure = {v_score:.3f}  (n = {n})")


CDR H1: V-Measure = 0.289  (n = 798)
CDR H2: V-Measure = 0.348  (n = 572)
CDR L1: V-Measure = 0.480  (n = 798)
CDR L2: V-Measure = 0.000  (n = 798)
CDR L3: V-Measure = 0.328  (n = 789)
